# NEH on NEH

In [16]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL_original/epoch_10.pth"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"
# CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_reduced_split_val_20.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================
def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.6f}")
print(f"{'AUC':<30} | {auc:.6f}")

print(f"{'F1':<30} | {f1:.6f}")
print(f"{'F2 (β=2)':<30} | {f2:.6f}")
print(f"{'Macro F1':<30} | {macro_f1:.6f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.6f}")


print(f"{'Precision':<30} | {precision:.6f}")
print(f"{'Recall':<30} | {recall:.6f}")

print(f"{'Macro Precision':<30} | {macro_precision:.6f}")
print(f"{'Macro Recall':<30} | {macro_recall:.6f}")

print(f"{'Specificity':<30} | {specificity:.6f}")
print(f"{'NPV':<30} | {npv:.6f}")

print(f"{'Cohen Kappa':<30} | {kappa:.6f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.6f}")
print("=" * 60)

# ============================================================
# EDL EVALUATION — epoch_3.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.8839
# AUC                            | 0.9537
# F1                             | 0.8841
# Macro F1                       | 0.8839
# Macro Precision                | 0.8849
# Macro Recall                   | 0.8848
# Specificity                    | 0.8559
# NPV                            | 0.9135
# Cohen Kappa                    | 0.7680

# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.243
# Singleton (%) (τ=0.9)          | 75.70
# ECE                            | 0.0172
# ============================================================


--- Loading EDL Model: epoch_10.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 21/21 [00:02<00:00,  8.97it/s]



EDL EVALUATION — epoch_10.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.873065
AUC                            | 0.950628
F1                             | 0.867742
F2 (β=2)                       | 0.873028
Macro F1                       | 0.872859
Macro F2 (β=2)                 | 0.872719
Precision                      | 0.876221
Recall                         | 0.859425
Macro Precision                | 0.873214
Macro Recall                   | 0.872655
Specificity                    | 0.885886
NPV                            | 0.870206
Cohen Kappa                    | 0.745740
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.147
Singleton (%) (τ=0.9)          | 85.29
ECE                            | 0.052900


# NEH on UCSD

In [ ]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_EDL/epoch_10.pth"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================
def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_10.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 3014/3014 [04:35<00:00, 10.93it/s]



EDL EVALUATION — epoch_10.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7430
AUC                            | 0.9421
F1                             | 0.7978
F2 (β=2)                       | 0.8889
Macro F1                       | 0.7227
Macro F2 (β=2)                 | 0.7437
Precision                      | 0.6815
Recall                         | 0.9621
Macro Precision                | 0.8017
Macro Recall                   | 0.7305
Specificity                    | 0.4990
NPV                            | 0.9219
Cohen Kappa                    | 0.4722
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.305
Singleton (%) (τ=0.9)          | 69.48
ECE                            | 0.1390


In [1]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_EDL/epoch_25.pth"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_25.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 3014/3014 [06:05<00:00,  8.26it/s]



EDL EVALUATION — epoch_25.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.6863
AUC                            | 0.9128
F1                             | 0.7651
F2 (β=2)                       | 0.6606
Macro F1                       | 0.6464
Macro F2 (β=2)                 | 0.6483
Precision                      | 0.6319
Recall                         | 0.9696
Macro Precision                | 0.7741
Macro Recall                   | 0.6701
Specificity                    | 0.3706
NPV                            | 0.9162
Cohen Kappa                    | 0.3509
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.212
Singleton (%) (τ=0.9)          | 78.79
ECE                            | 0.2236


# NEH on DHU

In [15]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL_original/epoch_10.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================
def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    # bin_ids = np.digitize(conf, bins) - 1
    bin_ids = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)


    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)

# ============================================================
# EDL EVALUATION — epoch_23.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.6785
# AUC                            | 0.9728
# F1                             | 0.7590
# F2 (β=2)                       | 0.6481
# Macro F1                       | 0.6383
# Macro F2 (β=2)                 | 0.6438
# Precision                      | 0.6136
# Recall                         | 0.9945
# Macro Precision                | 0.7988
# Macro Recall                   | 0.6728
# Specificity                    | 0.3511
# NPV                            | 0.9841
# Cohen Kappa                    | 0.3495
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.367
# Singleton (%) (τ=0.9)          | 63.28
# ECE                            | 0.1845
# ============================================================

# ============================================================
# EDL EVALUATION — epoch_10.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.7033
# AUC                            | 0.9719
# F1                             | 0.7739
# F2 (β=2)                       | 0.6773
# Macro F1                       | 0.6712
# Macro F2 (β=2)                 | 0.6733
# Precision                      | 0.6321
# Recall                         | 0.9979
# Macro Precision                | 0.8134
# Macro Recall                   | 0.6980
# Specificity                    | 0.3980
# NPV                            | 0.9947
# Cohen Kappa                    | 0.4002
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.333
# Singleton (%) (τ=0.9)          | 66.74
# ECE                            | 0.1755
# ============================================================

# ============================================================
# EDL EVALUATION — epoch_10.pth
# ============================================================
# Metric                         | Value
# ------------------------------------------------------------
# Accuracy                       | 0.9400
# AUC                            | 0.9809
# F1                             | 0.9409
# F2 (β=2)                       | 0.9400
# Macro F1                       | 0.9399
# Macro F2 (β=2)                 | 0.9400
# Precision                      | 0.9422
# Recall                         | 0.9396
# Macro Precision                | 0.9399
# Macro Recall                   | 0.9400
# Specificity                    | 0.9403
# NPV                            | 0.9376
# Cohen Kappa                    | 0.8799
# ------------------------------------------------------------
# Set Size (avg, τ=0.9)          | 1.234
# Singleton (%) (τ=0.9)          | 76.58
# ECE                            | 0.0376
# ============================================================


--- Loading EDL Model: epoch_10.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 90/90 [00:08<00:00, 10.47it/s]


EDL EVALUATION — epoch_10.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8105
AUC                            | 0.9742
F1                             | 0.8392
F2 (β=2)                       | 0.8032
Macro F1                       | 0.8042
Macro F2 (β=2)                 | 0.8012
Precision                      | 0.7384
Recall                         | 0.9719
Macro Precision                | 0.8475
Macro Recall                   | 0.8075
Specificity                    | 0.6432
NPV                            | 0.9567
Cohen Kappa                    | 0.6186
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.408
Singleton (%) (τ=0.9)          | 59.16
ECE                            | 0.0613


In [1]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_EDL/epoch_25.pth"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_25.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 90/90 [00:12<00:00,  6.96it/s]


EDL EVALUATION — epoch_25.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.6967
AUC                            | 0.9502
F1                             | 0.7660
F2 (β=2)                       | 0.6734
Macro F1                       | 0.6676
Macro F2 (β=2)                 | 0.6696
Precision                      | 0.6306
Recall                         | 0.9753
Macro Precision                | 0.7858
Macro Recall                   | 0.6916
Specificity                    | 0.4080
NPV                            | 0.9410
Cohen Kappa                    | 0.3871
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.239
Singleton (%) (τ=0.9)          | 76.06
ECE                            | 0.2066


# NEH on OCT C8

In [ ]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL/epoch_3.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered_imbalanced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================
def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_3.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 270/270 [00:24<00:00, 10.85it/s]


EDL EVALUATION — epoch_3.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8340
AUC                            | 0.9054
F1                             | 0.8819
F2 (β=2)                       | 0.8324
Macro F1                       | 0.8014
Macro F2 (β=2)                 | 0.7951
Precision                      | 0.8579
Recall                         | 0.9073
Macro Precision                | 0.8149
Macro Recall                   | 0.7917
Specificity                    | 0.6761
NPV                            | 0.7720
Cohen Kappa                    | 0.6035
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.276
Singleton (%) (τ=0.9)          | 72.38
ECE                            | 0.0585


: 

In [1]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_onlyvalM_5e4_EDL/epoch_25.pth"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")



macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")


precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")


print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_25.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 180/180 [00:23<00:00,  7.60it/s]


EDL EVALUATION — epoch_25.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7068
AUC                            | 0.8208
F1                             | 0.7681
F2 (β=2)                       | 0.6918
Macro F1                       | 0.6847
Macro F2 (β=2)                 | 0.6836
Precision                      | 0.6555
Recall                         | 0.9273
Macro Precision                | 0.7544
Macro Recall                   | 0.6958
Specificity                    | 0.4643
NPV                            | 0.8532
Cohen Kappa                    | 0.3998
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.246
Singleton (%) (τ=0.9)          | 75.44
ECE                            | 0.1908



--- Loading MC Dropout Model: epoch_9.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout): 100%|██████████| 180/180 [08:24<00:00,  2.80s/it]



Applying Venn-Abers Calibration...

MC DROPOUT EVALUATION: epoch_9.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.5701          | 0.5722         
AUC                            | 0.6759          | 0.6733         
F1                             | 0.6617          | 0.6942         
F2 (β=2)                       | 0.5525          | 0.5297         
Macro F1                       | 0.5360          | 0.4912         
Macro F2 (β=2)                 | 0.5432          | 0.5154         
Precision                      | 0.5627          | 0.5547         
Recall                         | 0.8030          | 0.9273         
Macro Precision                | 0.5773          | 0.6247         
Macro Recall                   | 0.5585          | 0.5545         
Specificity                    | 0.3140          | 0.1818         
NPV                            | 0.5919         

In [ ]:

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL/epoch_3.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 2. EDL HELPERS
# =====================================================
def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

# =====================================================
# 3. RELIABILITY METRICS (AS PROVIDED)
# =====================================================
def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(conf[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)

    return float(ece)

# =====================================================
# 4. DATASET
# =====================================================
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL
# =====================================================
print(f"\n--- Loading EDL Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# =====================================================
# 6. INFERENCE
# =====================================================
probs_all, labels_all, unc_all = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="EDL Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)

        p_hat, alpha, _ = edl_forward(logits)

        uncertainty = 2.0 / alpha.sum(dim=1, keepdim=True)

        probs_all.extend(p_hat[:, 1].cpu().numpy())
        labels_all.extend(labels.numpy())
        unc_all.extend(uncertainty.cpu().numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= 0.5).astype(int)
y_unc = np.asarray(unc_all)

# =====================================================
# 7. CONFUSION MATRIX DERIVED METRICS
# =====================================================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

specificity = tn / (tn + fp + 1e-8)
npv = tn / (tn + fn + 1e-8)

# =====================================================
# 8. STANDARD METRICS
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

kappa = cohen_kappa_score(y_true, y_pred)

# =====================================================
# 9. RELIABILITY METRICS (τ = 0.9)
# =====================================================
conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 10. FINAL DISPLAY
# =====================================================
print("\n" + "=" * 60)
print(f"EDL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {specificity:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")

print(f"{'ECE':<30} | {ece:.4f}")
print("=" * 60)



--- Loading EDL Model: epoch_3.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
EDL Inference: 100%|██████████| 435/435 [00:57<00:00,  7.60it/s]


EDL EVALUATION — epoch_3.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.7776
AUC                            | 0.9142
F1                             | 0.8099
Macro F1                       | 0.7711
Macro Precision                | 0.8105
Macro Recall                   | 0.7764
Specificity                    | 0.6127
NPV                            | 0.9098
Cohen Kappa                    | 0.5542
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.311
Singleton (%) (τ=0.9)          | 68.86
ECE                            | 0.1039


In [1]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e5_EDL"

IMG_SIZE = 512
BATCH_SIZE = 32
NUM_CLASSES = 2
EPOCH_RANGE = (7, 22)

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    m = models.resnet34(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

# =====================================================
# EDL FUNCTIONS
# =====================================================
def dirichlet_kl(alpha):
    S = torch.sum(alpha, dim=1, keepdim=True)
    ones = torch.ones_like(alpha)
    t1 = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    t2 = torch.sum((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S)), dim=1, keepdim=True)
    return t1 + t2

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

# =====================================================
# INFERENCE LOOP
# =====================================================
print("\n=========== EDL INFERENCE ===========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["state_dict"])
    model.eval()

    probs_all, labels_all = [], []
    evidences_all, correct_flags = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            p_hat, alpha, evidence = edl_forward(logits)

            probs = p_hat[:, 1]
            preds = (probs >= 0.5).long()

            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            evidences_all.extend(evidence.sum(dim=1).cpu().numpy())
            correct_flags.extend((preds == labels).cpu().numpy())

    y_true = np.array(labels_all)
    y_prob = np.array(probs_all)
    y_pred = (y_prob >= 0.5).astype(int)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = compute_ece(y_prob, y_true)
    err_unc = entropy(y_prob).mean() * 100
    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ EDL inference completed successfully.")



=========== EDL INFERENCE ===========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is e

Epoch 07 | F1=0.8709 | MacroF1=0.8610 | Acc=0.8617 | AUC=0.9341 | Spec=0.7963 | PrecM=0.8679 | RecM=0.8612 | NPV=0.9139 | Kappa=0.7232 | ECE=0.3307 | ErrUnc=48.04% | Set=1 | Singleton=8.19%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 08 | F1=0.8430 | MacroF1=0.8154 | Acc=0.8195 | AUC=0.9474 | Spec=0.6749 | PrecM=0.8481 | RecM=0.8185 | NPV=0.9459 | Kappa=0.6383 | ECE=0.3102 | ErrUnc=46.18% | Set=1 | Singleton=36.47%


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8211 | MacroF1=0.7777 | Acc=0.7861 | AUC=0.9457 | Spec=0.5954 | PrecM=0.8336 | RecM=0.7847 | NPV=0.9575 | Kappa=0.5710 | ECE=0.3087 | ErrUnc=41.88% | Set=1 | Singleton=46.04%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.7903 | MacroF1=0.7201 | Acc=0.7377 | AUC=0.9431 | Spec=0.4904 | PrecM=0.8121 | RecM=0.7359 | NPV=0.9627 | Kappa=0.4734 | ECE=0.3100 | ErrUnc=41.02% | Set=1 | Singleton=49.15%


Epoch 21 | F1=0.7991 | MacroF1=0.7380 | Acc=0.7522 | AUC=0.9454 | Spec=0.5231 | PrecM=0.8173 | RecM=0.7505 | NPV=0.9590 | Kappa=0.5028 | ECE=0.3052 | ErrUnc=41.73% | Set=1 | Singleton=47.67%

✅ EDL inference completed successfully.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL"

IMG_SIZE = 512
BATCH_SIZE = 32
NUM_CLASSES = 2
EPOCH_RANGE = (7, 22)

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    m = models.resnet34(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

# =====================================================
# EDL FUNCTIONS
# =====================================================
def dirichlet_kl(alpha):
    S = torch.sum(alpha, dim=1, keepdim=True)
    ones = torch.ones_like(alpha)
    t1 = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    t2 = torch.sum((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S)), dim=1, keepdim=True)
    return t1 + t2

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

# =====================================================
# INFERENCE LOOP
# =====================================================
print("\n=========== EDL INFERENCE ===========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["state_dict"])
    model.eval()

    probs_all, labels_all = [], []
    evidences_all, correct_flags = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            p_hat, alpha, evidence = edl_forward(logits)

            probs = p_hat[:, 1]
            preds = (probs >= 0.5).long()

            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            evidences_all.extend(evidence.sum(dim=1).cpu().numpy())
            correct_flags.extend((preds == labels).cpu().numpy())

    y_true = np.array(labels_all)
    y_prob = np.array(probs_all)
    y_pred = (y_prob >= 0.5).astype(int)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = compute_ece(y_prob, y_true)
    err_unc = entropy(y_prob).mean() * 100
    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ EDL inference completed successfully.")


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_5e5_EDL"

IMG_SIZE = 512
BATCH_SIZE = 32
NUM_CLASSES = 2
EPOCH_RANGE =(3, 22)

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    m = models.resnet34(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

# =====================================================
# EDL FUNCTIONS
# =====================================================
def dirichlet_kl(alpha):
    S = torch.sum(alpha, dim=1, keepdim=True)
    ones = torch.ones_like(alpha)
    t1 = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    t2 = torch.sum((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S)), dim=1, keepdim=True)
    return t1 + t2

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

# =====================================================
# INFERENCE LOOP
# =====================================================
print("\n=========== EDL INFERENCE ===========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["state_dict"])
    model.eval()

    probs_all, labels_all = [], []
    evidences_all, correct_flags = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            p_hat, alpha, evidence = edl_forward(logits)

            probs = p_hat[:, 1]
            preds = (probs >= 0.5).long()

            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            evidences_all.extend(evidence.sum(dim=1).cpu().numpy())
            correct_flags.extend((preds == labels).cpu().numpy())

    y_true = np.array(labels_all)
    y_prob = np.array(probs_all)
    y_pred = (y_prob >= 0.5).astype(int)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = compute_ece(y_prob, y_true)
    err_unc = entropy(y_prob).mean() * 100
    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ EDL inference completed successfully.")


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



=========== EDL INFERENCE ===========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 03 | F1=0.7674 | MacroF1=0.6741 | Acc=0.7008 | AUC=0.9504 | Spec=0.4176 | PrecM=0.7920 | RecM=0.6987 | NPV=0.9533 | Kappa=0.3991 | ECE=0.3063 | ErrUnc=47.61% | Set=1 | Singleton=30.54%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.7205 | MacroF1=0.5828 | Acc=0.6283 | AUC=0.8901 | Spec=0.3004 | PrecM=0.7193 | RecM=0.6258 | NPV=0.8586 | Kappa=0.2529 | ECE=0.3338 | ErrUnc=38.99% | Set=1 | Singleton=47.06%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.7040 | MacroF1=0.5174 | Acc=0.5895 | AUC=0.8969 | Spec=0.2044 | PrecM=0.7097 | RecM=0.5867 | NPV=0.8665 | Kappa=0.1744 | ECE=0.3493 | ErrUnc=36.62% | Set=1 | Singleton=53.38%


Epoch 21 | F1=0.7095 | MacroF1=0.5349 | Acc=0.6005 | AUC=0.8852 | Spec=0.2267 | PrecM=0.7184 | RecM=0.5977 | NPV=0.8771 | Kappa=0.1965 | ECE=0.3546 | ErrUnc=35.87% | Set=1 | Singleton=55.49%

✅ EDL inference completed successfully.


In [3]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# CONFIG
# =====================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered_split_test_15.csv"
CKPT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_5e4_EDL"

IMG_SIZE = 512
BATCH_SIZE = 32
NUM_CLASSES = 2
EPOCH_RANGE = (3, 22)

# =====================================================
# DATASET
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        return self.transform(img), label

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# MODEL
# =====================================================
def get_model():
    m = models.resnet34(pretrained=False)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

# =====================================================
# EDL FUNCTIONS
# =====================================================
def dirichlet_kl(alpha):
    S = torch.sum(alpha, dim=1, keepdim=True)
    ones = torch.ones_like(alpha)
    t1 = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    t2 = torch.sum((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S)), dim=1, keepdim=True)
    return t1 + t2

def edl_forward(logits):
    evidence = F.softplus(logits)
    alpha = evidence + 1
    S = alpha.sum(dim=1, keepdim=True)
    p_hat = alpha / S
    return p_hat, alpha, evidence

def entropy(p):
    return -p * np.log(p + 1e-8) - (1 - p) * np.log(1 - p + 1e-8)

def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        acc = np.mean(labels[mask] == (probs[mask] >= 0.5))
        conf = np.mean(probs[mask])
        ece += (mask.sum() / len(probs)) * abs(acc - conf)
    return ece

# =====================================================
# METRICS
# =====================================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

# =====================================================
# INFERENCE LOOP
# =====================================================
print("\n=========== EDL INFERENCE ===========\n")

for epoch in range(EPOCH_RANGE[0], EPOCH_RANGE[1]):

    ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}.pth")
    if not os.path.exists(ckpt_path):
        continue

    model = get_model()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["state_dict"])
    model.eval()

    probs_all, labels_all = [], []
    evidences_all, correct_flags = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(imgs)
            p_hat, alpha, evidence = edl_forward(logits)

            probs = p_hat[:, 1]
            preds = (probs >= 0.5).long()

            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            evidences_all.extend(evidence.sum(dim=1).cpu().numpy())
            correct_flags.extend((preds == labels).cpu().numpy())

    y_true = np.array(labels_all)
    y_prob = np.array(probs_all)
    y_pred = (y_prob >= 0.5).astype(int)

    f1 = f1_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    spec = specificity(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro")
    rec_m = recall_score(y_true, y_pred, average="macro")
    npv_val = npv(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    ece = compute_ece(y_prob, y_true)
    err_unc = entropy(y_prob).mean() * 100
    singleton = (y_prob > 0.9).mean() * 100

    print(
        f"Epoch {epoch:02d} | "
        f"F1={f1:.4f} | MacroF1={macro_f1:.4f} | "
        f"Acc={acc:.4f} | AUC={auc:.4f} | "
        f"Spec={spec:.4f} | "
        f"PrecM={prec_m:.4f} | RecM={rec_m:.4f} | "
        f"NPV={npv_val:.4f} | "
        f"Kappa={kappa:.4f} | "
        f"ECE={ece:.4f} | "
        f"ErrUnc={err_unc:.2f}% | "
        f"Set=1 | Singleton={singleton:.2f}%"
    )

print("\n✅ EDL inference completed successfully.")


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



=========== EDL INFERENCE ===========



/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 03 | F1=0.8482 | MacroF1=0.8374 | Acc=0.8381 | AUC=0.9171 | Spec=0.7773 | PrecM=0.8431 | RecM=0.8377 | NPV=0.8824 | Kappa=0.6759 | ECE=0.3482 | ErrUnc=36.39% | Set=1 | Singleton=37.50%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


...

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 19 | F1=0.8320 | MacroF1=0.8208 | Acc=0.8215 | AUC=0.8989 | Spec=0.7649 | PrecM=0.8255 | RecM=0.8211 | NPV=0.8599 | Kappa=0.6426 | ECE=0.3932 | ErrUnc=27.60% | Set=1 | Singleton=44.30%


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 20 | F1=0.8161 | MacroF1=0.8089 | Acc=0.8092 | AUC=0.8889 | Spec=0.7775 | PrecM=0.8103 | RecM=0.8089 | NPV=0.8275 | Kappa=0.6181 | ECE=0.3864 | ErrUnc=29.08% | Set=1 | Singleton=39.68%


Epoch 21 | F1=0.7350 | MacroF1=0.6115 | Acc=0.6507 | AUC=0.8629 | Spec=0.3354 | PrecM=0.7452 | RecM=0.6484 | NPV=0.8954 | Kappa=0.2982 | ECE=0.3954 | ErrUnc=26.90% | Set=1 | Singleton=64.84%

✅ EDL inference completed successfully.
